In [65]:
def load_swc(file_path):
    """
    加载SWC文件并返回DataFrame
    """
    data = []
    with open(file_path, 'r', encoding='ISO-8859-1') as f:
        for line in f:
            if line.startswith('#') or line.strip() == '':
                continue
            parts = line.strip().split()
            data.append([int(parts[0]), int(parts[1]), float(parts[2]), float(parts[3]), float(parts[4]), 
                         float(parts[5]), int(parts[6])])
    df = pd.DataFrame(data, columns=['id', 'type', 'x', 'y', 'z', 'radius', 'parent'])
    return df


def get_second_largest_component(df):
    """
    获取第二大连通块
    """
    G = nx.Graph()
    
    # 创建图，将每个节点和其父节点之间添加边
    for _, row in df.iterrows():
        if row['parent'] != -1:  # 跳过无父节点的行
            G.add_edge(row['id'], row['parent'])
    
    # 获取所有连通块
    components = list(nx.connected_components(G))
    
    # 按连通块的大小排序
    components_sorted = sorted(components, key=len, reverse=True)
    
    # 获取第二大连通块（如果存在的话）
    if len(components_sorted) > 1:
        second_largest_component = components_sorted[1]
    else:
        second_largest_component = None  # 如果没有第二大连通块，则返回None
    return df[df['id'].isin(second_largest_component)]

def save_final_swc(merged_df, output_file):
    """
    保存合并后的SWC文件
    """
    merged_df.to_csv(output_file, sep=' ', header=False, index=False)
    print(f"最终处理后的 SWC 文件已保存至：{output_file}")

# 使用示例
main_tree_file =r"D:\GXQ\cross_em_lm\test_data\reconnect\50799931696_sort.swc"
swc_file =r"D:\GXQ\cross_em_lm\test_data\reconnect\50799931696.swc"
output_file = r"D:\GXQ\cross_em_lm\test_data\reconnect\50799931696_sub.swc"

# 加载 SWC 数据
main_df = load_swc(main_tree_file)
swc_df = load_swc(swc_file)
sub_tree_df=get_second_largest_component(swc_df)

print(len(main_df))
#sub_tree_node_id, main_tree_node_id=calculate_closest_points(main_df,sub_tree_df)
print(int(sub_tree_node_id))
save_final_swc(sub_tree_df,output_file)

1377
1384
最终处理后的 SWC 文件已保存至：D:\GXQ\cross_em_lm\test_data\reconnect\50799931696_sub.swc


In [70]:
import pandas as pd
import numpy as np
from scipy.spatial import distance_matrix

def read_swc(filename):
    nodes = {}
    with open(filename, 'r') as f:
        for line in f:
            print(f)
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split()
            if len(parts) != 7:
                print(f"警告：行格式不正确，已忽略：{line}")
                continue
            n = int(parts[0])
            T = int(parts[1])
            x = float(parts[2])
            y = float(parts[3])
            z = float(parts[4])
            radius = float(parts[5])
            P = int(float(parts[6]))
            nodes[n] = {'n': n, 'T': T, 'x': x, 'y': y, 'z': z, 'radius': radius, 'P': P, 'children': []}

    for node in nodes.values():
        P = node['P']
        if P != -1 and P in nodes:
            nodes[P]['children'].append(node['n'])
        elif P != -1:
            print(f"警告：父节点{P}不存在，节点{node['n']}的父节点设为-1")
            node['P'] = -1
    return nodes





def calculate_closest_points(main_df, sub_tree_df):
    """
    计算主树和子树空间最接近的点
    """
    # 获取主树和子树的坐标
    main_coords = main_df[['x', 'y', 'z']].to_numpy()
    sub_tree_coords = sub_tree_df[['x', 'y', 'z']].to_numpy()

    # 计算主树和子树所有节点之间的欧几里得距离
    dist_matrix = distance_matrix(sub_tree_coords, main_coords)

    # 找到最小的距离
    min_dist_index = np.unravel_index(np.argmin(dist_matrix), dist_matrix.shape)
    sub_tree_node_id = sub_tree_df.iloc[min_dist_index[0]]['n']  # 使用 'n' 而不是 'id'
    main_tree_node_id = main_df.iloc[min_dist_index[1]]['n']    # 使用 'n' 而不是 'id'
    min_distance = dist_matrix[min_dist_index]

    print(f"最小距离的节点对：子树节点 {sub_tree_node_id} 与主树节点 {main_tree_node_id}，距离 = {min_distance:.4f}")

    return sub_tree_node_id, main_tree_node_id



def save_to_swc(df, filename):
    """
    将DataFrame保存为标准SWC格式文件
    """
    with open(filename, 'w') as f:
        f.write("# 标准化后的SWC文件\n")
        for idx, row in df.iterrows():
            n = int(row['n'])  # 节点ID
            T = int(row['T'])  # 节点类型
            x = float(row['x'])  # x坐标
            y = float(row['y'])  # y坐标
            z = float(row['z'])  # z坐标
            radius = float(row['radius'])  # 半径
            P = int(row['P'])  # 父节点ID
            f.write(f"{n} {T} {x} {y} {z} {radius} {P}\n")
    print(f"已保存为 {filename}")

def assign_new_ids_to_subtree(sub_tree_df, main_df):
    """
    为子树节点分配新的ID，并更新父节点信息。
    """
    # 获取主树的最大ID，以便为子树节点分配新ID
    max_main_id = main_df['n'].max()

    # 创建一个新的DataFrame，用于存储更新后的子树节点信息
    updated_sub_tree_df = sub_tree_df.copy()

    # 为子树节点分配新的ID
    id_mapping = {}
    for idx, row in updated_sub_tree_df.iterrows():
        new_id = max_main_id + 1
        updated_sub_tree_df.at[idx, 'n'] = new_id
        id_mapping[row['n']] = new_id
        max_main_id = new_id  # 更新最大ID

    # 更新父节点信息，配对子树节点的父节点更改为新ID
    updated_sub_tree_df['P'] = updated_sub_tree_df['P'].map(id_mapping).fillna(updated_sub_tree_df['P'])

    return updated_sub_tree_df, id_mapping




    return main_df
def update_soma_node(sub_tree_df, sub_tree_node_id, main_tree_node_id):
    """
    找到子树的胞体节点，将父节点P修改为主树节点，并将type修改为0
    """
    # 找到子树的胞体节点，通常胞体节点的类型T是1（可以根据需要调整）
    soma_node_idx = sub_tree_df[sub_tree_df['n'] == sub_tree_node_id]
    
    if not soma_node_idx.empty:
        # 更新胞体节点的父节点为主树节点
        sub_tree_df.loc[sub_tree_df['n'] == sub_tree_node_id, 'P'] = main_tree_node_id
        
        # 更新胞体节点的type为0
        sub_tree_df.loc[sub_tree_df['n'] == sub_tree_node_id, 'T'] = 0
        
        print(f"胞体节点 {sub_tree_node_id} 的父节点已更新为主树节点 {main_tree_node_id}，type 已改为 0.")
    else:
        print(f"未找到子树胞体节点 {sub_tree_node_id}.")

    return sub_tree_df


def update_soma_nodes(updated_sub_tree_df,main_tree_node_id):
    """
    将子树中胞体节点的父节点ID改为配对的主树节点ID，并将胞体节点的类型T改为0。
    
    参数:
    updated_sub_tree_df: 更新后的子树DataFrame
    sub_tree_node_id: 子树中胞体节点的ID
    main_tree_node_id: 配对的主树节点ID
    """
    # 找到子树中的胞体节点（父节点 P == -1）
    soma_nodes = updated_sub_tree_df[updated_sub_tree_df['P'] == -1]
    
    # 遍历每个胞体节点
    for idx, row in soma_nodes.iterrows():
        soma_node_id = row['n']  # 获取胞体节点ID
        
        # 更新胞体节点的父节点为主树节点ID
        updated_sub_tree_df.at[idx, 'P'] = main_tree_node_id
        
        # 将胞体节点的类型T更新为0
        updated_sub_tree_df.at[idx, 'T'] = 0

        print(f"胞体节点 {soma_node_id} 的父节点ID已更新为主树节点 {main_tree_node_id}，类型已更新为0")
    
    return updated_sub_tree_df



# 示例使用方法
if __name__ == "__main__":
    # 读取并调整数据
    sub_tree_df= read_swc( r"D:\GXQ\cross_em_lm\test_data\reconnect\39490508916_sub.swc")
   
    #write_swc(nodes, r"D:\GXQ\cross_em_lm\test_data\reconnect\updated_sub_tree11.swc")
    # 将字典类型的 nodes 转换为 DataFrame
    #main_df = read_swc(r"D:\GXQ\cross_em_lm\test_data\reconnect\50799931696_sort.swc")
    main_df = pd.DataFrame.from_dict(main_df, orient='index')
    main_df.reset_index(drop=True, inplace=True)  # 重设索引，确保顺序

    # 处理 nodes，将其转换为 DataFrame
    sub_tree_df = pd.DataFrame.from_dict(nodes, orient='index')
    
    sub_tree_df.reset_index(drop=True, inplace=True)  # 重设索引，确保顺序

    #计算配对节点
    sub_tree_node_id, main_tree_node_id = calculate_closest_points(main_df, sub_tree_df)
    # soma_node = 18138
    nodes = adjust_soma_and_roots(nodes, int(sub_tree_node_id))
    # #子树reassign id
    # updated_sub_tree_df, id_mapping =assign_new_ids_to_subtree(sub_tree_df, main_df)
    # #重连
    # updated_sub_tree_df = update_soma_nodes(updated_sub_tree_df, main_tree_node_id)
    # hh_main_df = pd.concat([main_df, updated_sub_tree_df], ignore_index=True)

    # save_to_swc(hh_main_df,output_file = r"D:\GXQ\cross_em_lm\test_data\reconnect\50799931696_reconnect.swc")



<_io.TextIOWrapper name='D:\\GXQ\\cross_em_lm\\test_data\\reconnect\\39490508916_sub.swc' mode='r' encoding='UTF-8'>
<_io.TextIOWrapper name='D:\\GXQ\\cross_em_lm\\test_data\\reconnect\\39490508916_sub.swc' mode='r' encoding='UTF-8'>
<_io.TextIOWrapper name='D:\\GXQ\\cross_em_lm\\test_data\\reconnect\\39490508916_sub.swc' mode='r' encoding='UTF-8'>
<_io.TextIOWrapper name='D:\\GXQ\\cross_em_lm\\test_data\\reconnect\\39490508916_sub.swc' mode='r' encoding='UTF-8'>
<_io.TextIOWrapper name='D:\\GXQ\\cross_em_lm\\test_data\\reconnect\\39490508916_sub.swc' mode='r' encoding='UTF-8'>
<_io.TextIOWrapper name='D:\\GXQ\\cross_em_lm\\test_data\\reconnect\\39490508916_sub.swc' mode='r' encoding='UTF-8'>
<_io.TextIOWrapper name='D:\\GXQ\\cross_em_lm\\test_data\\reconnect\\39490508916_sub.swc' mode='r' encoding='UTF-8'>
<_io.TextIOWrapper name='D:\\GXQ\\cross_em_lm\\test_data\\reconnect\\39490508916_sub.swc' mode='r' encoding='UTF-8'>
<_io.TextIOWrapper name='D:\\GXQ\\cross_em_lm\\test_data\\reconn

TypeError: 'numpy.ndarray' object is not callable

1377
18138
最终处理后的 SWC 文件已保存至：D:\GXQ\cross_em_lm\test_data\reconnect\50799931696_sub.swc


In [75]:


def find_path(nodes, start_node, end_node):
    queue = deque([(start_node, [start_node])])
    visited = {start_node}
    while queue:
        current_node, path = queue.popleft()
        if current_node == end_node:
            return path
        for neighbor in nodes.get(current_node, {}).get('children', []):
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append((neighbor, path + [neighbor]))
    return None


def reverse_path(nodes, path):
    path_len = len(path)
    for i in range(path_len - 1):
        child_id = path[i]
        parent_id = path[i + 1]
        nodes[child_id]['P'] = parent_id
    for node_id in path:
        nodes[node_id]['children'] = []
    for i in range(path_len - 1):
        parent_id = path[i + 1]
        child_id = path[i]
        nodes[parent_id]['children'].append(child_id)


def adjust_soma_and_roots(nodes,soma_node):
    # 使用加权后的胞体检测方法
    # soma_node = find_potential_soma_with_adjusted_weights(nodes)
    # print(f"选择节点 {soma_node} 作为新的胞体节点")

    # 找到原始的 P=-1 节点（根节点）
    original_soma_nodes = [node['n'] for node in nodes.values() if node['P'] == -1 and node['n'] != soma_node]
    #print(f"原始 P=-1 节点: {original_soma_nodes}")

    # 处理每个原始根节点到新胞体节点的路径
    for old_soma_node in original_soma_nodes:
        path = find_path(nodes, old_soma_node, soma_node)
        if path:
            #print(f"从旧胞体节点 {old_soma_node} 到新胞体节点 {soma_node} 的路径: {path}")
            reverse_path(nodes, path)
        else:
            print(f"无法找到从旧胞体节点 {old_soma_node} 到新胞体节点 {soma_node} 的路径")

    # 设置新胞体节点的类型和父节点
    nodes[soma_node]['T'] = 1
    nodes[soma_node]['P'] = -1
    for node in nodes.values():
        if node['n'] != soma_node:
            node['T'] = 0  # 非胞体节点的类型设为1

    # 重建父子关系
    for node in nodes.values():
        node['children'] = []
    for node in nodes.values():
        P = node['P']
        if P != -1 and P in nodes:
            nodes[P]['children'].append(node['n'])
        elif P != -1:
            print(f"警告：父节点 {P} 不存在，节点 {node['n']} 的父节点设为 -1")
            node['P'] = -1
    return nodes

def write_swc(nodes, filename):
    with open(filename, 'w') as f:
        f.write("# 标准化后的SWC文件\n")
        for n in sorted(nodes.keys()):
            node = nodes[n]
            f.write(f"{node['n']} {node['T']} {node['x']} {node['y']} {node['z']} {node['radius']} {node['P']}\n")

sub_tree_df= read_swc( r"D:\GXQ\cross_em_lm\test_data\reconnect\39490508916_sub.swc")
nodes=adjust_soma_and_roots(sub_tree_df,18138)
write_swc(nodes,r"D:\GXQ\cross_em_lm\test_data\reconnect\39490508916_sub123.swc")

<_io.TextIOWrapper name='D:\\GXQ\\cross_em_lm\\test_data\\reconnect\\39490508916_sub.swc' mode='r' encoding='UTF-8'>
<_io.TextIOWrapper name='D:\\GXQ\\cross_em_lm\\test_data\\reconnect\\39490508916_sub.swc' mode='r' encoding='UTF-8'>
<_io.TextIOWrapper name='D:\\GXQ\\cross_em_lm\\test_data\\reconnect\\39490508916_sub.swc' mode='r' encoding='UTF-8'>
<_io.TextIOWrapper name='D:\\GXQ\\cross_em_lm\\test_data\\reconnect\\39490508916_sub.swc' mode='r' encoding='UTF-8'>
<_io.TextIOWrapper name='D:\\GXQ\\cross_em_lm\\test_data\\reconnect\\39490508916_sub.swc' mode='r' encoding='UTF-8'>
<_io.TextIOWrapper name='D:\\GXQ\\cross_em_lm\\test_data\\reconnect\\39490508916_sub.swc' mode='r' encoding='UTF-8'>
<_io.TextIOWrapper name='D:\\GXQ\\cross_em_lm\\test_data\\reconnect\\39490508916_sub.swc' mode='r' encoding='UTF-8'>
<_io.TextIOWrapper name='D:\\GXQ\\cross_em_lm\\test_data\\reconnect\\39490508916_sub.swc' mode='r' encoding='UTF-8'>
<_io.TextIOWrapper name='D:\\GXQ\\cross_em_lm\\test_data\\reconn